# IMDb medallion pipeline (Delta Live Tables)

Source code for the Delta Live Tables pipeline that builds the IMDb warehouse.
It is not run cell by cell: attach this notebook as the source of a DLT pipeline and start the pipeline.

| Layer | Tables | What happens |
|---|---|---|
| Bronze | 7 `brz_*` | Raw TSV files loaded as-is, tagged with `_source_file` |
| Silver | 7 `slv_*` | Types standardised and 60 `expect_all_or_drop` rules applied |
| Gold | 6 `dim_*`, 3 `fact_*`, 3 `bridge_*` | Star schema with surrogate keys and audit columns |

Source files are read from `/Volumes/workspace/imdb_dw/data`.

## Bronze layer
One table per source file, read with header and tab separator. No transformations.

In [0]:
# ============================================================
# DLT PIPELINE: BRONZE LAYER
# ============================================================

import dlt
from pyspark.sql.functions import lit, current_timestamp

DATA_PATH = "/Volumes/workspace/imdb_dw/data"

# ============================================================
# BRONZE: name.basics
# ============================================================
@dlt.table(
    name="brz_name_basics",
    comment="Bronze: Raw name.basics data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_name_basics():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/name.basics.tsv")
        .withColumn("_source_file", lit("name.basics.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.basics
# ============================================================
@dlt.table(
    name="brz_title_basics",
    comment="Bronze: Raw title.basics data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_basics():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.basics.tsv")
        .withColumn("_source_file", lit("title.basics.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.akas
# ============================================================
@dlt.table(
    name="brz_title_akas",
    comment="Bronze: Raw title.akas data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_akas():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.akas.tsv")
        .withColumn("_source_file", lit("title.akas.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.crew
# ============================================================
@dlt.table(
    name="brz_title_crew",
    comment="Bronze: Raw title.crew data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_crew():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.crew.tsv")
        .withColumn("_source_file", lit("title.crew.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.episode
# ============================================================
@dlt.table(
    name="brz_title_episode",
    comment="Bronze: Raw title.episode data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_episode():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.episode.tsv")
        .withColumn("_source_file", lit("title.episode.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.principals
# ============================================================
@dlt.table(
    name="brz_title_principals",
    comment="Bronze: Raw title.principals data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_principals():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.principals.tsv")
        .withColumn("_source_file", lit("title.principals.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

# ============================================================
# BRONZE: title.ratings
# ============================================================
@dlt.table(
    name="brz_title_ratings",
    comment="Bronze: Raw title.ratings data from IMDb",
    table_properties={"quality": "bronze"}
)
def brz_title_ratings():
    return (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("inferSchema", "true")
        .option("nullValue", "\\N")
        .csv(f"{DATA_PATH}/title.ratings.tsv")
        .withColumn("_source_file", lit("title.ratings.tsv"))
        .withColumn("_load_timestamp", current_timestamp())
    )

## Silver layer
Cleans each bronze table and enforces data quality with `@dlt.expect_all_or_drop`: 60 rules in total, covering keys, ID formats, year ranges and allowed values. Rows that fail a rule are dropped and counted in the pipeline's event log.

In [0]:
# ============================================================
# DLT PIPELINE: SILVER LAYER - COMPREHENSIVE VALIDATIONS
# ============================================================
# Based on:
# - Professor's DQX approach
# - Kimball Data Warehouse Toolkit
# - Industry best practices
#
# Validation Categories:
# 1. Completeness Checks (NOT NULL, NOT EMPTY)
# 2. Format Checks (Pattern, Length, Prefix)
# 3. Range Checks (Min, Max, Between)
# 4. Domain Checks (Valid values, Enumerations)
# 5. Cross-Field Checks (Business rules)
#
# Total Validations: 59
# Total Transformations: 6
# ============================================================

import dlt
from pyspark.sql.functions import col, trim, when, lit, current_timestamp, length

# ============================================================
# SILVER: slv_name_basics
# ============================================================
# Source: brz_name_basics (14,903,875 rows)
# Primary Key: nconst
# 
# VALIDATIONS (8):
# - Completeness: nconst NOT NULL (PK), primaryName NOT NULL
# - Format: nconst starts with 'nm', length 9-10
# - Range: birthYear 1000-2025, deathYear 1000-2025
# - Cross-Field: deathYear >= birthYear (except BC dates)
#
# Expected Drop: ~95 rows (NULL name, invalid death dates)
# ============================================================

@dlt.table(
    name="slv_name_basics",
    comment="Silver: Cleaned name.basics - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_nconst_not_null": "nconst IS NOT NULL",
    "completeness_nconst_not_empty": "LENGTH(TRIM(nconst)) > 0",
    "completeness_primaryName_not_null": "primaryName IS NOT NULL",
    
    # Format Checks
    "format_nconst_prefix": "nconst LIKE 'nm%'",
    "format_nconst_length": "LENGTH(nconst) BETWEEN 9 AND 10",
    
    # Range Checks
    "range_birthYear_valid": "birthYear IS NULL OR (birthYear >= 1000 AND birthYear <= 2025)",
    "range_deathYear_valid": "deathYear IS NULL OR (deathYear >= 1000 AND deathYear <= 2025)",
    
    # Cross-Field Checks
    "crossfield_death_after_birth": "deathYear IS NULL OR birthYear IS NULL OR deathYear >= birthYear OR birthYear < 1000"
})
def slv_name_basics():
    return (
        dlt.read("brz_name_basics")
        .withColumn("nconst", trim(col("nconst")))
        .withColumn("primaryName", trim(col("primaryName")))
        .withColumn("primaryProfession", trim(col("primaryProfession")))
        .withColumn("knownForTitles", trim(col("knownForTitles")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_basics
# ============================================================
# Source: brz_title_basics (12,095,664 rows)
# Primary Key: tconst
#
# VALIDATIONS (7):
# - Completeness: tconst NOT NULL (PK), titleType, primaryTitle NOT NULL
# - Format: tconst starts with 'tt', length 9-10
# - Domain: titleType in valid list
#
# TRANSFORMATIONS (4):
# - Invalid isAdult values (years) → 0
# - Invalid startYear → NULL
# - Invalid endYear → NULL
# - Invalid runtimeMinutes → NULL
#
# Expected Drop: 0 rows
# ============================================================

@dlt.table(
    name="slv_title_basics",
    comment="Silver: Cleaned title.basics - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_tconst_not_null": "tconst IS NOT NULL",
    "completeness_tconst_not_empty": "LENGTH(TRIM(tconst)) > 0",
    "completeness_titleType_not_null": "titleType IS NOT NULL",
    "completeness_primaryTitle_not_null": "primaryTitle IS NOT NULL",
    
    # Format Checks
    "format_tconst_prefix": "tconst LIKE 'tt%'",
    "format_tconst_length": "LENGTH(tconst) BETWEEN 9 AND 10",
    
    # Domain Checks
    "domain_titleType_valid": "titleType IN ('movie', 'short', 'tvEpisode', 'tvMiniSeries', 'tvMovie', 'tvPilot', 'tvSeries', 'tvShort', 'tvSpecial', 'video', 'videoGame')"
})
def slv_title_basics():
    return (
        dlt.read("brz_title_basics")
        .withColumn("tconst", trim(col("tconst")))
        .withColumn("titleType", trim(col("titleType")))
        .withColumn("primaryTitle", trim(col("primaryTitle")))
        .withColumn("originalTitle", trim(col("originalTitle")))
        .withColumn("genres", trim(col("genres")))
        # Transformation: Fix invalid isAdult (years → 0)
        .withColumn("isAdult",
            when(col("isAdult") == 1, 1)
            .when(col("isAdult") == 0, 0)
            .otherwise(0)
        )
        # Transformation: Fix invalid startYear
        .withColumn("startYear",
            when((col("startYear") >= 1800) & (col("startYear") <= 2030), col("startYear"))
            .otherwise(None)
        )
        # Transformation: Fix invalid endYear
        .withColumn("endYear",
            when((col("endYear") >= 1800) & (col("endYear") <= 2030), col("endYear"))
            .otherwise(None)
        )
        # Transformation: Fix invalid runtimeMinutes
        .withColumn("runtimeMinutes",
            when(col("runtimeMinutes").cast("int") > 0, col("runtimeMinutes").cast("int"))
            .otherwise(None)
        )
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_akas
# ============================================================
# Source: brz_title_akas (54,124,127 rows)
# Primary Key: titleId + ordering (Composite)
# Foreign Key: titleId → title.basics.tconst
#
# VALIDATIONS (9):
# - Completeness: titleId, ordering, title, isOriginalTitle NOT NULL
# - Format: titleId starts with 'tt', length 9-10
# - Range: ordering >= 1
# - Domain: isOriginalTitle in (0, 1)
#
# TRANSFORMATIONS (2):
# - NULL region → "UNKNOWN"
# - NULL language → "UNKNOWN"
#
# Expected Drop: ~531 rows (NULL isOriginalTitle)
# ============================================================

@dlt.table(
    name="slv_title_akas",
    comment="Silver: Cleaned title.akas - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_titleId_not_null": "titleId IS NOT NULL",
    "completeness_titleId_not_empty": "LENGTH(TRIM(titleId)) > 0",
    "completeness_ordering_not_null": "ordering IS NOT NULL",
    "completeness_title_not_null": "title IS NOT NULL",
    "completeness_isOriginalTitle_not_null": "isOriginalTitle IS NOT NULL",
    
    # Format Checks
    "format_titleId_prefix": "titleId LIKE 'tt%'",
    "format_titleId_length": "LENGTH(titleId) BETWEEN 9 AND 10",
    
    # Range Checks
    "range_ordering_valid": "ordering >= 1",
    
    # Domain Checks
    "domain_isOriginalTitle_valid": "isOriginalTitle IN (0, 1)"
})
def slv_title_akas():
    return (
        dlt.read("brz_title_akas")
        .withColumn("titleId", trim(col("titleId")))
        .withColumn("title", trim(col("title")))
        # Transformation: NULL region → "UNKNOWN"
        .withColumn("region",
            when(col("region").isNotNull() & (trim(col("region")) != ""), trim(col("region")))
            .otherwise(lit("UNKNOWN"))
        )
        # Transformation: NULL language → "UNKNOWN"
        .withColumn("language",
            when(col("language").isNotNull() & (trim(col("language")) != ""), trim(col("language")))
            .otherwise(lit("UNKNOWN"))
        )
        .withColumn("types", trim(col("types")))
        .withColumn("attributes", trim(col("attributes")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_crew
# ============================================================
# Source: brz_title_crew (12,095,664 rows)
# Primary Key: tconst
# Foreign Key: tconst → title.basics.tconst
#
# VALIDATIONS (4):
# - Completeness: tconst NOT NULL (PK)
# - Format: tconst starts with 'tt', length 9-10
#
# Note: directors/writers are optional (44%/49% NULL - expected)
#
# Expected Drop: 0 rows
# ============================================================

@dlt.table(
    name="slv_title_crew",
    comment="Silver: Cleaned title.crew - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_tconst_not_null": "tconst IS NOT NULL",
    "completeness_tconst_not_empty": "LENGTH(TRIM(tconst)) > 0",
    
    # Format Checks
    "format_tconst_prefix": "tconst LIKE 'tt%'",
    "format_tconst_length": "LENGTH(tconst) BETWEEN 9 AND 10"
})
def slv_title_crew():
    return (
        dlt.read("brz_title_crew")
        .withColumn("tconst", trim(col("tconst")))
        .withColumn("directors", trim(col("directors")))
        .withColumn("writers", trim(col("writers")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_episode
# ============================================================
# Source: brz_title_episode (9,324,110 rows)
# Primary Key: tconst
# Foreign Keys:
#   - tconst → title.basics.tconst (Episode)
#   - parentTconst → title.basics.tconst (TV Series)
#
# VALIDATIONS (11):
# - Completeness: tconst, parentTconst NOT NULL
# - Format: tconst/parentTconst start with 'tt', length 9-10
# - Range: seasonNumber >= 0, episodeNumber >= 0 (if not NULL)
# - Cross-Field: tconst != parentTconst (episode not its own parent)
#
# Note: seasonNumber/episodeNumber optional (20% NULL - specials)
#
# Expected Drop: 0 rows
# ============================================================

@dlt.table(
    name="slv_title_episode",
    comment="Silver: Cleaned title.episode - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_tconst_not_null": "tconst IS NOT NULL",
    "completeness_tconst_not_empty": "LENGTH(TRIM(tconst)) > 0",
    "completeness_parentTconst_not_null": "parentTconst IS NOT NULL",
    "completeness_parentTconst_not_empty": "LENGTH(TRIM(parentTconst)) > 0",
    
    # Format Checks
    "format_tconst_prefix": "tconst LIKE 'tt%'",
    "format_tconst_length": "LENGTH(tconst) BETWEEN 9 AND 10",
    "format_parentTconst_prefix": "parentTconst LIKE 'tt%'",
    "format_parentTconst_length": "LENGTH(parentTconst) BETWEEN 9 AND 10",
    
    # Range Checks
    "range_seasonNumber_valid": "seasonNumber IS NULL OR seasonNumber >= 0",
    "range_episodeNumber_valid": "episodeNumber IS NULL OR episodeNumber >= 0",
    
    # Cross-Field Checks
    "crossfield_not_self_parent": "tconst != parentTconst"
})
def slv_title_episode():
    return (
        dlt.read("brz_title_episode")
        .withColumn("tconst", trim(col("tconst")))
        .withColumn("parentTconst", trim(col("parentTconst")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_principals
# ============================================================
# Source: brz_title_principals (96,132,477 rows)
# Primary Key: tconst + ordering + nconst (Composite)
# Foreign Keys:
#   - tconst → title.basics.tconst
#   - nconst → name.basics.nconst
#
# VALIDATIONS (12):
# - Completeness: tconst, ordering, nconst, category NOT NULL
# - Format: tconst starts with 'tt', nconst starts with 'nm'
# - Range: ordering >= 1
# - Domain: category in valid list
#
# Note: job/characters optional (81%/51% NULL - expected)
#
# Expected Drop: 0 rows
# ============================================================

@dlt.table(
    name="slv_title_principals",
    comment="Silver: Cleaned title.principals - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_tconst_not_null": "tconst IS NOT NULL",
    "completeness_tconst_not_empty": "LENGTH(TRIM(tconst)) > 0",
    "completeness_ordering_not_null": "ordering IS NOT NULL",
    "completeness_nconst_not_null": "nconst IS NOT NULL",
    "completeness_nconst_not_empty": "LENGTH(TRIM(nconst)) > 0",
    "completeness_category_not_null": "category IS NOT NULL",
    
    # Format Checks
    "format_tconst_prefix": "tconst LIKE 'tt%'",
    "format_tconst_length": "LENGTH(tconst) BETWEEN 9 AND 10",
    "format_nconst_prefix": "nconst LIKE 'nm%'",
    "format_nconst_length": "LENGTH(nconst) BETWEEN 9 AND 10",
    
    # Range Checks
    "range_ordering_valid": "ordering >= 1",
    
    # Domain Checks
    "domain_category_valid": "category IN ('actor', 'actress', 'archive_footage', 'archive_sound', 'casting_director', 'cinematographer', 'composer', 'director', 'editor', 'producer', 'production_designer', 'self', 'writer')"
})
def slv_title_principals():
    return (
        dlt.read("brz_title_principals")
        .withColumn("tconst", trim(col("tconst")))
        .withColumn("nconst", trim(col("nconst")))
        .withColumn("category", trim(col("category")))
        .withColumn("job", trim(col("job")))
        .withColumn("characters", trim(col("characters")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )


# ============================================================
# SILVER: slv_title_ratings
# ============================================================
# Source: brz_title_ratings (1,605,649 rows)
# Primary Key: tconst
# Foreign Key: tconst → title.basics.tconst
#
# VALIDATIONS (9):
# - Completeness: tconst, averageRating, numVotes NOT NULL
# - Format: tconst starts with 'tt', length 9-10
# - Range: averageRating 1.0-10.0, numVotes >= 5
#
# Expected Drop: 0 rows
# ============================================================

@dlt.table(
    name="slv_title_ratings",
    comment="Silver: Cleaned title.ratings - Valid records only",
    table_properties={"quality": "silver"}
)
@dlt.expect_all_or_drop({
    # Completeness Checks
    "completeness_tconst_not_null": "tconst IS NOT NULL",
    "completeness_tconst_not_empty": "LENGTH(TRIM(tconst)) > 0",
    "completeness_averageRating_not_null": "averageRating IS NOT NULL",
    "completeness_numVotes_not_null": "numVotes IS NOT NULL",
    
    # Format Checks
    "format_tconst_prefix": "tconst LIKE 'tt%'",
    "format_tconst_length": "LENGTH(tconst) BETWEEN 9 AND 10",
    
    # Range Checks
    "range_averageRating_min": "averageRating >= 1.0",
    "range_averageRating_max": "averageRating <= 10.0",
    "range_numVotes_min": "numVotes >= 5"
})
def slv_title_ratings():
    return (
        dlt.read("brz_title_ratings")
        .withColumn("tconst", trim(col("tconst")))
        .withColumn("DI_JOB_ID", lit("SILVER_LOAD"))
        .withColumn("DI_LOAD_DT", current_timestamp())
    )

## Gold layer
Dimensional model: dimensions first, then facts, then bridge tables for the many-to-many relationships (title↔genre, person↔profession, title↔alternate titles).

In [0]:
# ============================================================
# DLT PIPELINE: GOLD LAYER - DIMENSIONAL MODEL (FIXED)
# ============================================================

import dlt
from pyspark.sql.functions import (
    col, trim, when, lit, current_timestamp,
    explode, split, monotonically_increasing_id
)

# ============================================================
# ===================== DIMENSIONS ===========================
# ============================================================

@dlt.table(
    name="dim_title",
    comment="Gold: Title Dimension",
    table_properties={"quality": "gold"}
)
def dim_title():
    return (
        dlt.read("slv_title_basics")
        .select(
            monotonically_increasing_id().alias("TitleKey"),
            col("tconst").alias("Tconst"),
            col("titleType").alias("TitleType"),
            col("primaryTitle").alias("PrimaryTitle"),
            col("originalTitle").alias("OriginalTitle"),
            col("isAdult").alias("IsAdult"),
            col("startYear").alias("StartYear"),
            col("endYear").alias("EndYear"),
            col("runtimeMinutes").alias("RuntimeMinutes"),
            col("genres").alias("Genres"),
            lit("dim_title").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="dim_name",
    comment="Gold: Name/Person Dimension",
    table_properties={"quality": "gold"}
)
def dim_name():
    return (
        dlt.read("slv_name_basics")
        .select(
            monotonically_increasing_id().alias("NameKey"),
            col("nconst").alias("Nconst"),
            col("primaryName").alias("PrimaryName"),
            col("birthYear").alias("BirthYear"),
            col("deathYear").alias("DeathYear"),
            col("primaryProfession").alias("PrimaryProfession"),
            col("knownForTitles").alias("KnownForTitles"),
            lit("dim_name").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="dim_genre",
    comment="Gold: Genre Dimension",
    table_properties={"quality": "gold"}
)
def dim_genre():
    return (
        dlt.read("slv_title_basics")
        .filter(col("genres").isNotNull())
        .select(explode(split(col("genres"), ",")).alias("GenreName"))
        .select(trim(col("GenreName")).alias("GenreName"))
        .filter(col("GenreName") != "")
        .distinct()
        .withColumn("GenreKey", monotonically_increasing_id())
        .select(
            col("GenreKey"),
            col("GenreName"),
            lit("dim_genre").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="dim_profession",
    comment="Gold: Profession Dimension",
    table_properties={"quality": "gold"}
)
def dim_profession():
    return (
        dlt.read("slv_name_basics")
        .filter(col("primaryProfession").isNotNull())
        .select(explode(split(col("primaryProfession"), ",")).alias("ProfessionName"))
        .select(trim(col("ProfessionName")).alias("ProfessionName"))
        .filter(col("ProfessionName") != "")
        .distinct()
        .withColumn("ProfessionKey", monotonically_increasing_id())
        .select(
            col("ProfessionKey"),
            col("ProfessionName"),
            lit("dim_profession").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="dim_region",
    comment="Gold: Region Dimension",
    table_properties={"quality": "gold"}
)
def dim_region():
    return (
        dlt.read("slv_title_akas")
        .select(col("region").alias("RegionCode"))
        .distinct()
        .withColumn("RegionKey", monotonically_increasing_id())
        .withColumn("RegionName",
            when(col("RegionCode") == "US", "United States")
            .when(col("RegionCode") == "GB", "United Kingdom")
            .when(col("RegionCode") == "DE", "Germany")
            .when(col("RegionCode") == "FR", "France")
            .when(col("RegionCode") == "JP", "Japan")
            .when(col("RegionCode") == "IN", "India")
            .when(col("RegionCode") == "ES", "Spain")
            .when(col("RegionCode") == "IT", "Italy")
            .when(col("RegionCode") == "CA", "Canada")
            .when(col("RegionCode") == "AU", "Australia")
            .when(col("RegionCode") == "BR", "Brazil")
            .when(col("RegionCode") == "MX", "Mexico")
            .when(col("RegionCode") == "RU", "Russia")
            .when(col("RegionCode") == "CN", "China")
            .when(col("RegionCode") == "KR", "South Korea")
            .when(col("RegionCode") == "UNKNOWN", "Unknown")
            .otherwise(col("RegionCode"))
        )
        .select(
            col("RegionKey"),
            col("RegionCode"),
            col("RegionName"),
            lit("dim_region").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="dim_language",
    comment="Gold: Language Dimension",
    table_properties={"quality": "gold"}
)
def dim_language():
    return (
        dlt.read("slv_title_akas")
        .select(col("language").alias("LanguageCode"))
        .distinct()
        .withColumn("LanguageKey", monotonically_increasing_id())
        .withColumn("LanguageName",
            when(col("LanguageCode") == "en", "English")
            .when(col("LanguageCode") == "es", "Spanish")
            .when(col("LanguageCode") == "fr", "French")
            .when(col("LanguageCode") == "de", "German")
            .when(col("LanguageCode") == "ja", "Japanese")
            .when(col("LanguageCode") == "hi", "Hindi")
            .when(col("LanguageCode") == "it", "Italian")
            .when(col("LanguageCode") == "pt", "Portuguese")
            .when(col("LanguageCode") == "ru", "Russian")
            .when(col("LanguageCode") == "zh", "Chinese")
            .when(col("LanguageCode") == "ko", "Korean")
            .when(col("LanguageCode") == "ar", "Arabic")
            .when(col("LanguageCode") == "cmn", "Mandarin Chinese")
            .when(col("LanguageCode") == "yue", "Cantonese")
            .when(col("LanguageCode") == "UNKNOWN", "Unknown")
            .otherwise(col("LanguageCode"))
        )
        .select(
            col("LanguageKey"),
            col("LanguageCode"),
            col("LanguageName"),
            lit("dim_language").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


# ============================================================
# ===================== FACTS ================================
# ============================================================

@dlt.table(
    name="fact_title_ratings",
    comment="Gold: Title Ratings Fact",
    table_properties={"quality": "gold"}
)
def fact_title_ratings():
    df_ratings = dlt.read("slv_title_ratings")
    df_title = dlt.read("dim_title")
    
    return (
        df_ratings
        .join(df_title, df_ratings.tconst == df_title.Tconst, "inner")
        .select(
            monotonically_increasing_id().alias("RatingKey"),
            df_title.TitleKey.alias("TitleKey"),
            df_ratings.tconst.alias("Tconst"),
            df_ratings.averageRating.alias("AverageRating"),
            df_ratings.numVotes.alias("NumVotes"),
            lit("fact_title_ratings").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="fact_episodes",
    comment="Gold: Episodes Fact",
    table_properties={"quality": "gold"}
)
def fact_episodes():
    df_episode = dlt.read("slv_title_episode")
    df_title = dlt.read("dim_title")
    
    # Create aliases for joins to avoid ambiguity
    df_parent = df_title.select(
        col("TitleKey").alias("ParentTitleKey"),
        col("Tconst").alias("ParentJoinKey")
    )
    
    df_episode_title = df_title.select(
        col("TitleKey").alias("EpisodeTitleKey"),
        col("Tconst").alias("EpisodeJoinKey")
    )
    
    return (
        df_episode
        .join(df_parent, df_episode.parentTconst == df_parent.ParentJoinKey, "inner")
        .join(df_episode_title, df_episode.tconst == df_episode_title.EpisodeJoinKey, "inner")
        .select(
            monotonically_increasing_id().alias("EpisodeKey"),
            col("ParentTitleKey"),
            col("EpisodeTitleKey"),
            df_episode.tconst.alias("Tconst"),
            df_episode.parentTconst.alias("ParentTconst"),
            df_episode.seasonNumber.alias("SeasonNumber"),
            df_episode.episodeNumber.alias("EpisodeNumber"),
            lit("fact_episodes").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="fact_title_principals",
    comment="Gold: Title Principals Fact",
    table_properties={"quality": "gold"}
)
def fact_title_principals():
    df_principals = dlt.read("slv_title_principals")
    df_title = dlt.read("dim_title")
    df_name = dlt.read("dim_name")
    
    return (
        df_principals
        .join(df_title, df_principals.tconst == df_title.Tconst, "inner")
        .join(df_name, df_principals.nconst == df_name.Nconst, "inner")
        .select(
            monotonically_increasing_id().alias("PrincipalKey"),
            df_title.TitleKey.alias("TitleKey"),
            df_name.NameKey.alias("NameKey"),
            df_principals.tconst.alias("Tconst"),
            df_principals.nconst.alias("Nconst"),
            df_principals.ordering.alias("Ordering"),
            df_principals.category.alias("Category"),
            df_principals.job.alias("Job"),
            df_principals.characters.alias("Characters"),
            lit("fact_title_principals").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


# ============================================================
# ===================== BRIDGES ==============================
# ============================================================

@dlt.table(
    name="bridge_title_genre",
    comment="Gold: Title-Genre Bridge",
    table_properties={"quality": "gold"}
)
def bridge_title_genre():
    df_title = dlt.read("dim_title")
    df_genre = dlt.read("dim_genre")
    df_basics = dlt.read("slv_title_basics").filter(col("genres").isNotNull())
    
    df_exploded = df_basics.select(
        col("tconst").alias("src_tconst"),
        explode(split(col("genres"), ",")).alias("genre")
    ).withColumn("genre", trim(col("genre")))
    
    return (
        df_exploded
        .join(df_title, df_exploded.src_tconst == df_title.Tconst, "inner")
        .join(df_genre, df_exploded.genre == df_genre.GenreName, "inner")
        .select(
            monotonically_increasing_id().alias("TitleGenreKey"),
            df_title.TitleKey.alias("TitleKey"),
            df_genre.GenreKey.alias("GenreKey"),
            df_exploded.src_tconst.alias("Tconst"),
            df_genre.GenreName.alias("GenreName"),
            lit("bridge_title_genre").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="bridge_name_profession",
    comment="Gold: Name-Profession Bridge",
    table_properties={"quality": "gold"}
)
def bridge_name_profession():
    df_name = dlt.read("dim_name")
    df_profession = dlt.read("dim_profession")
    df_basics = dlt.read("slv_name_basics").filter(col("primaryProfession").isNotNull())
    
    df_exploded = df_basics.select(
        col("nconst").alias("src_nconst"),
        explode(split(col("primaryProfession"), ",")).alias("profession")
    ).withColumn("profession", trim(col("profession")))
    
    return (
        df_exploded
        .join(df_name, df_exploded.src_nconst == df_name.Nconst, "inner")
        .join(df_profession, df_exploded.profession == df_profession.ProfessionName, "inner")
        .select(
            monotonically_increasing_id().alias("NameProfessionKey"),
            df_name.NameKey.alias("NameKey"),
            df_profession.ProfessionKey.alias("ProfessionKey"),
            df_exploded.src_nconst.alias("Nconst"),
            df_profession.ProfessionName.alias("ProfessionName"),
            lit("bridge_name_profession").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )


@dlt.table(
    name="bridge_title_akas",
    comment="Gold: Title-Akas Bridge",
    table_properties={"quality": "gold"}
)
def bridge_title_akas():
    df_akas = dlt.read("slv_title_akas")
    df_title = dlt.read("dim_title")
    df_region = dlt.read("dim_region")
    df_language = dlt.read("dim_language")
    
    return (
        df_akas
        .join(df_title, df_akas.titleId == df_title.Tconst, "inner")
        .join(df_region, df_akas.region == df_region.RegionCode, "left")
        .join(df_language, df_akas.language == df_language.LanguageCode, "left")
        .select(
            monotonically_increasing_id().alias("TitleAkaKey"),
            df_title.TitleKey.alias("TitleKey"),
            df_region.RegionKey.alias("RegionKey"),
            df_language.LanguageKey.alias("LanguageKey"),
            df_akas.titleId.alias("Tconst"),
            df_akas.title.alias("AkaTitle"),
            df_akas.ordering.alias("Ordering"),
            df_akas.types.alias("Types"),
            df_akas.attributes.alias("Attributes"),
            df_akas.isOriginalTitle.alias("IsOriginalTitle"),
            lit("bridge_title_akas").alias("DI_JOB_ID"),
            current_timestamp().alias("DI_LOAD_DT")
        )
    )